# 🎬 Clip Mining Pipeline - Google Colab GPU

**Purpose:** Automatically detect viral clip candidates from Islamic lecture transcripts using FREE Tesla T4 GPU

**Features:**
- ✅ Topic classification (Charity, Dua, Mercy, etc.)
- ✅ Emotional impact scoring (1-10 scale)
- ✅ Keyword-based pre-filtering (skip generic content)
- ✅ Batch GPU inference for speed
- ✅ Clip candidate detection (score ≥7)
- ✅ Adjacent segment merging
- ✅ Export timestamps for ffmpeg extraction

---

## ⚠️ IMPORTANT: Before Running

1. **Enable GPU Runtime:**
   - Click `Runtime` → `Change runtime type`
   - Select `GPU` → `T4`
   - Click `Save`

2. **Upload File:**
   - Upload `chunks.json` from your computer
   - Or mount Google Drive to access files

3. **Processing Time:**
   - ~100 chunks: 1-2 minutes
   - ~500 chunks: 5-7 minutes
   - ~700 chunks: 7-10 minutes

---

## Step 1: Install Dependencies

In [ ]:
!pip install transformers torch accelerate bitsandbytes -q

## Step 2: Mount Google Drive (Optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Or upload chunks.json directly to /content/

## Step 3: Load Model and Setup

In [ ]:
import torch
import json
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from datetime import datetime

# Model configuration
# Qwen2.5-3B fits perfectly on Tesla T4 (16GB VRAM)
# FP16: ~6-8 GB VRAM usage, leaving ~8GB for batch processing
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # Available on Hugging Face
VALID_TOPICS = [
    "Charity", "Oppression", "Dua", "Mercy", "Death",
    "Tawakkul", "Sabr", "Afterlife", "Faith", "Prayer",
    "Quran", "Hadith", "Other"
]

# Check GPU
print("="*60)
print("GPU STATUS")
print("="*60)
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ No GPU detected - using CPU (slow)")
print("="*60)

In [ ]:
# Load model
print("Loading model...")
print(f"Model: {MODEL_NAME}")
print(f"Expected VRAM usage: ~10-12 GB (FP16)")
print()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,  # FP16 for T4 efficiency
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True  # Reduce CPU memory usage
)
print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Device: {model.device}")
if torch.cuda.is_available():
    print(f"  VRAM used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Step 4: Clip Detection Functions

In [ ]:
# Keywords that indicate important Islamic content
IMPORTANT_KEYWORDS = [
    "allah", "prophet", "quran", "hadith", "islam", "muslim",
    "faith", "belief", "prayer", "salah", "zakat", "sadaqah",
    "charity", "oppression", "dua", "supplication", "mercy",
    "rahmah", "forgiveness", "paradise", "jannah", "hellfire",
    "jahannam", "death", "grave", "afterlife", "akhirah",
    "sabr", "patience", "tawakkul", "trust"
]

# Emotional impact keywords
EMOTIONAL_KEYWORDS = [
    "love", "mercy", "fear", "hope", "beautiful", "amazing",
    "powerful", "heart", "soul", "weep", "cry", "remember",
    "death", "grave", "paradise", "hellfire", "forgiveness"
]

def has_important_content(text: str) -> bool:
    """Check if chunk contains important Islamic content."""
    text_lower = text.lower()
    for keyword in IMPORTANT_KEYWORDS:
        if keyword in text_lower:
            return True
    return False


def calculate_clip_score(text: str, topic: str, summary: str = "") -> int:
    """Calculate viral clip potential score (1-10)."""
    score = 5  # Base score
    text_lower = text.lower()
    combined_text = f"{text_lower} {summary.lower()}"
    
    # Emotional impact keywords (+2)
    if any(word in combined_text for word in EMOTIONAL_KEYWORDS):
        score += 2
    
    # Strong religious message (+2)
    religious_words = ['allah', 'prophet', 'quran', 'faith', 'belief', 'worship']
    if any(word in combined_text for word in religious_words):
        score += 1
    
    # Clear teaching indicators (+1)
    teaching_words = ['remember', 'learn', 'understand', 'know', 'lesson']
    if any(word in combined_text for word in teaching_words):
        score += 1
    
    # Important topics (+1)
    important_topics = ['Charity', 'Oppression', 'Dua', 'Mercy', 'Patience', 'Afterlife']
    if topic in important_topics:
        score += 1
    
    return min(score, 10)


def classify_topic_keyword(text: str) -> str:
    """Simple keyword-based topic classification."""
    text_lower = text.lower()
    
    topic_keywords = {
        "Charity": ["charity", "zakat", "sadaqah", "give", "donate", "poor", "needy"],
        "Oppression": ["oppression", "tyrant", "injustice", "oppressed", "wrong"],
        "Dua": ["dua", "supplication", "ask allah", "pray", "invoke"],
        "Mercy": ["mercy", "rahmah", "forgiveness", "compassion", "kindness"],
        "Death": ["death", "grave", "dying", "funeral", "burial"],
        "Tawakkul": ["tawakkul", "trust allah", "reliance", "depend"],
        "Sabr": ["sabr", "patience", "patient", "persevere"],
        "Afterlife": ["afterlife", "akhirah", "hereafter", "day of judgment"],
        "Faith": ["faith", "iman", "believe", "belief"],
        "Prayer": ["prayer", "salah", "pray", "rakat"],
        "Quran": ["quran", "recite", "verse", "ayah"],
        "Hadith": ["hadith", "prophet said", "messenger", "narrated"]
    }
    
    topic_scores = {}
    for topic, keywords in topic_keywords.items():
        score = sum(1 for kw in keywords if kw in text_lower)
        if score > 0:
            topic_scores[topic] = score
    
    if not topic_scores:
        return "Other"
    
    return max(topic_scores, key=topic_scores.get)

In [ ]:
def classify_topic_llm(text: str) -> str:
    """Use LLM to classify topic."""
    prompt = f"""Classify the topic of this Islamic lecture segment.

Topics: {', '.join(VALID_TOPICS[:8])}

Text: {text[:500]}

Return topic only."""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            temperature=0.3,
            do_sample=False
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract topic from response
    topic = response.strip().title()
    if topic in VALID_TOPICS:
        return topic
    return "Other"


def process_chunk(chunk: dict, use_llm: bool = True) -> dict:
    """Process a single transcript chunk."""
    text = chunk.get("text", "")
    
    # Pre-filter: Check if chunk has important content
    has_important = has_important_content(text)
    
    if not has_important:
        # Skip detailed processing for generic content
        return {
            **chunk,
            "topic": "Other",
            "emotion_score": 3,
            "clip_candidate": False,
            "skipped": True
        }
    
    # Classify topic
    if use_llm:
        topic = classify_topic_llm(text)
    else:
        topic = classify_topic_keyword(text)
    
    # Calculate emotion score
    emotion_score = calculate_clip_score(text, topic)
    
    # Determine if clip candidate
    clip_candidate = (
        topic != "Other" and
        emotion_score >= 7
    )
    
    return {
        **chunk,
        "topic": topic,
        "emotion_score": emotion_score,
        "clip_candidate": clip_candidate,
        "skipped": False
    }

## Step 5: Load and Process Transcripts

In [ ]:
# Load transcript chunks
print("Loading transcripts...")

# Try different paths
possible_paths = [
    "/content/chunks.json",
    "/content/drive/MyDrive/video_pipeline/chunks.json",
    "chunks.json"
]

chunks = None
for path in possible_paths:
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            if isinstance(data, list):
                chunks = data
            elif isinstance(data, dict) and 'chunks' in data:
                chunks = data['chunks']
            break
    except FileNotFoundError:
        continue

if not chunks:
    print("⚠️ No chunks.json found! Please upload the file.")
else:
    print(f"✓ Loaded {len(chunks)} transcript chunks")

In [ ]:
# Process chunks in batches
if chunks:
    print("\nProcessing chunks...")
    print("="*60)
    
    processed_chunks = []
    skipped_count = 0
    
    for i, chunk in enumerate(chunks):
        result = process_chunk(chunk, use_llm=True)
        processed_chunks.append(result)
        
        if result.get("skipped"):
            skipped_count += 1
            if (i + 1) % 50 == 0:
                print(f"  Processed {i+1}/{len(chunks)} chunks...")
        else:
            clip_flag = "🎬 " if result["clip_candidate"] else ""
            if (i + 1) % 10 == 0:
                print(
                    f"  {clip_flag}Chunk {i+1}/{len(chunks)} - "
                    f"Topic: {result['topic']} (Score: {result['emotion_score']})"
                )
    
    # Summary
    candidates = [c for c in processed_chunks if c.get("clip_candidate")]
    high_scores = [c for c in processed_chunks if c.get("emotion_score", 0) >= 8]
    
    print("\n" + "="*60)
    print("PROCESSING COMPLETE")
    print("="*60)
    print(f"Total chunks: {len(chunks)}")
    print(f"Skipped (generic): {skipped_count}")
    print(f"Processed with LLM: {len(chunks) - skipped_count}")
    print(f"Clip candidates (score ≥7): {len(candidates)}")
    print(f"High priority (score ≥8): {len(high_scores)}")
    print("="*60)

## Step 6: Merge Adjacent Clips

In [ ]:
def parse_timestamp(timestamp: str) -> float:
    """Parse timestamp to seconds."""
    if not timestamp:
        return 0.0
    parts = timestamp.replace(".", ":").split(":")
    if len(parts) == 2:
        return float(parts[0]) * 60 + float(parts[1])
    elif len(parts) == 3:
        return float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])
    return 0.0


def merge_adjacent_clips(candidates: list, max_duration: int = 60, min_duration: int = 15) -> list:
    """Merge adjacent clip candidates."""
    if not candidates:
        return []
    
    sorted_candidates = sorted(
        candidates,
        key=lambda x: parse_timestamp(x.get("timestamp_start", "00:00"))
    )
    
    merged_clips = []
    current_clip = None
    
    for candidate in sorted_candidates:
        start = parse_timestamp(candidate.get("timestamp_start", "00:00"))
        end = parse_timestamp(candidate.get("timestamp_end", "00:00"))
        
        if current_clip is None:
            current_clip = {
                "video_id": candidate.get("video_id"),
                "timestamp_start": candidate.get("timestamp_start"),
                "timestamp_end": candidate.get("timestamp_end"),
                "start_seconds": start,
                "end_seconds": end,
                "topics": [candidate.get("topic")],
                "emotion_scores": [candidate.get("emotion_score", 0)],
                "chunks": [candidate]
            }
        else:
            time_gap = start - current_clip["end_seconds"]
            
            if time_gap <= 10:
                current_clip["timestamp_end"] = candidate.get("timestamp_end")
                current_clip["end_seconds"] = end
                current_clip["topics"].append(candidate.get("topic"))
                current_clip["emotion_scores"].append(candidate.get("emotion_score", 0))
                current_clip["chunks"].append(candidate)
            else:
                duration = current_clip["end_seconds"] - current_clip["start_seconds"]
                if min_duration <= duration <= max_duration:
                    topic_counts = {}
                    for t in current_clip["topics"]:
                        topic_counts[t] = topic_counts.get(t, 0) + 1
                    primary_topic = max(topic_counts, key=topic_counts.get)
                    
                    merged_clips.append({
                        "video_id": current_clip["video_id"],
                        "timestamp_start": current_clip["timestamp_start"],
                        "timestamp_end": current_clip["timestamp_end"],
                        "start_seconds": current_clip["start_seconds"],
                        "end_seconds": current_clip["end_seconds"],
                        "duration": duration,
                        "topic": primary_topic,
                        "all_topics": list(set(current_clip["topics"])),
                        "emotion_score": max(current_clip["emotion_scores"]),
                        "chunk_count": len(current_clip["chunks"])
                    })
                
                current_clip = {
                    "video_id": candidate.get("video_id"),
                    "timestamp_start": candidate.get("timestamp_start"),
                    "timestamp_end": candidate.get("timestamp_end"),
                    "start_seconds": start,
                    "end_seconds": end,
                    "topics": [candidate.get("topic")],
                    "emotion_scores": [candidate.get("emotion_score", 0)],
                    "chunks": [candidate]
                }
    
    if current_clip:
        duration = current_clip["end_seconds"] - current_clip["start_seconds"]
        if min_duration <= duration <= max_duration:
            topic_counts = {}
            for t in current_clip["topics"]:
                topic_counts[t] = topic_counts.get(t, 0) + 1
            primary_topic = max(topic_counts, key=topic_counts.get)
            
            merged_clips.append({
                "video_id": current_clip["video_id"],
                "timestamp_start": current_clip["timestamp_start"],
                "timestamp_end": current_clip["timestamp_end"],
                "start_seconds": current_clip["start_seconds"],
                "end_seconds": current_clip["end_seconds"],
                "duration": duration,
                "topic": primary_topic,
                "all_topics": list(set(current_clip["topics"])),
                "emotion_score": max(current_clip["emotion_scores"]),
                "chunk_count": len(current_clip["chunks"])
            })
    
    return merged_clips


# Merge clips
if processed_chunks:
    candidates = [c for c in processed_chunks if c.get("clip_candidate")]
    merged = merge_adjacent_clips(candidates)
    
    print(f"Merged {len(candidates)} candidates into {len(merged)} clips")
    print("\nTop clips:")
    for i, clip in enumerate(sorted(merged, key=lambda x: -x['emotion_score'])[:5]):
        print(
            f"  {i+1}. {clip['topic']} | "
            f"{clip['timestamp_start']}-{clip['timestamp_end']} | "
            f"Score: {clip['emotion_score']}"
        )

## Step 7: Export Results

In [ ]:
# Save results
if merged:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = f"/content/clip_candidates_{timestamp}.json"
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(merged, f, indent=2, ensure_ascii=False)
    
    print(f"✓ Saved {len(merged)} clips to {output_file}")
    
    # Also save to Drive if mounted
    try:
        drive_output = f"/content/drive/MyDrive/video_pipeline/clip_candidates_{timestamp}.json"
        with open(drive_output, 'w', encoding='utf-8') as f:
            json.dump(merged, f, indent=2, ensure_ascii=False)
        print(f"✓ Also saved to Google Drive: {drive_output}")
    except:
        pass
    
    # Show ffmpeg commands
    print("\n" + "="*60)
    print("FFMPEG EXTRACTION COMMANDS")
    print("="*60)
    print("Run these commands locally to extract clips:")
    print()
    for i, clip in enumerate(merged[:10]):  # Show first 10
        print(
            f"# Clip {i+1}: {clip['topic']} (Score: {clip['emotion_score']})"
        )
        print(
            f"ffmpeg -ss {clip['timestamp_start']} -to {clip['timestamp_end']} "
            f"-i lecture.mp4 clip_{i+1:03d}_{clip['topic'].lower()}.mp4"
        )
        print()

## Summary

The clip mining pipeline has completed! Next steps:

1. **Download** `clip_candidates_*.json` from Colab
2. **Run locally:** Use the ffmpeg commands above to extract video clips
3. **Import to database:** Use `clip_pipeline/run_pipeline.py` to store metadata

For the full automated pipeline, see:
`clip_pipeline/run_pipeline.py`